# PDF dataset timing -- vector classification on a sampled set

Every page is treated as its **vectorised** version (`conversion.convert_page_to_vector_text`: native text
removed and drawn back as vector glyph paths). The notebook:

1. scans a folder of PDFs and counts each page's vectors after vectorisation (`num_vectors_vectorised`),
2. graphs that distribution,
3. samples up to `NUM_SAMPLES` **distinct PDFs, one page each**, evenly by rank across the distribution,
4. runs `generate_pipeline_report` on them with one config file (`scripts/report_configs/classify_timing_v2.json`:
   P2 `stub`, P3 `v2_vector_class`, no benchmark, no debug images) so only vector classification is timed,
5. plots the time taken against `num_vectors_vectorised`.

Logic: `notebooks/_helpers/dataset_timing_helpers.py`. Outputs land under `outputs/pdf_dataset_timing/<run_tag>/`.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

PROJECT_ROOT = (
    Path.cwd().parents[0] if (Path.cwd() / "pdf_dataset_timing.ipynb").exists() else Path.cwd()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import hashlib
import json
import subprocess
from datetime import datetime

import matplotlib.pyplot as plt

from notebooks._helpers import dataset_timing_helpers as h
from raster_parser.commons.logging_setup import configure_logging, get_logger
from raster_parser.commons.paths import output_dir
from raster_parser.evaluation import dump_io
from raster_parser.evaluation.evaluate.timing import flatten_page_timing
from raster_parser.evaluation.report.report_config import ReportConfig

configure_logging()
_LOG = get_logger("pdf_dataset_timing")

## Parameters

In [ ]:
# Folder of PDFs to scan (searched recursively for *.pdf).
INPUT_FOLDER = PROJECT_ROOT / "references"

# ReportConfig JSON run on the sample -- only input_files/pages/input_dir/output_root are overridden below.
# Keep it classification-only: P2 stub, no benchmark, no debug images.
REPORT_CONFIG_PATH = PROJECT_ROOT / "scripts" / "report_configs" / "classify_timing_v2.json"

# Distinct PDFs to sample (one page each), evenly by rank of num_vectors_vectorised.
NUM_SAMPLES = 50
# Skip pages with more vectorised vectors than this (None = no cap). Classifying the very largest pages
# needs several GB of RAM (the 120k-vector page in references/ peaks near 7 GB).
MAX_VECTORS = None

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = output_dir("pdf_dataset_timing", RUN_TAG)

# The scan is cached per input folder (no content hashing): delete SCAN_DIR to force a rescan.
_SCAN_KEY = hashlib.sha1(str(Path(INPUT_FOLDER).resolve()).encode("utf-8")).hexdigest()[:16]
SCAN_DIR = output_dir("pdf_dataset_timing", "_scan_cache", _SCAN_KEY)

assert Path(INPUT_FOLDER).is_dir(), f"not a folder: {INPUT_FOLDER}"
assert Path(REPORT_CONFIG_PATH).is_file(), f"missing report config: {REPORT_CONFIG_PATH}"
print("input folder:", INPUT_FOLDER)
print("report config:", REPORT_CONFIG_PATH)
print("run dir:", RUN_DIR)

## 1. Scan the dataset (vectorised pages)

One record per page: the original native-text and vector counts plus `num_vectors_vectorised`. Cached in
`SCAN_DIR/page_stats_vectorised.txt`.

In [ ]:
stats_path = SCAN_DIR / "page_stats_vectorised.txt"
if stats_path.is_file():
    page_records = h.load_page_records(stats_path)
    print(
        f"reusing cached scan {stats_path} ({len(page_records)} page(s)) -- delete {SCAN_DIR} to rescan"
    )
else:
    pdf_paths = sorted(Path(INPUT_FOLDER).rglob("*.pdf"))
    print(f"found {len(pdf_paths)} PDF(s) under {INPUT_FOLDER}")
    page_records, skipped = [], []
    for i, pdf_path in enumerate(pdf_paths, 1):
        try:
            page_records.extend(h.scan_pdf(pdf_path))
        except Exception as exc:  # keep scanning the rest of the dataset
            skipped.append(pdf_path)
            _LOG.warning("skipping %s: %s", pdf_path, exc)
        if i % 25 == 0 or i == len(pdf_paths):
            print(f"  scanned {i}/{len(pdf_paths)} PDF(s), {len(page_records)} page(s) so far")
    h.save_page_records(page_records, stats_path)
    print(f"wrote {stats_path} ({len(skipped)} PDF(s) skipped)")
assert page_records, "no pages found"

## 2. Vector counts

Distribution of `num_vectors_vectorised`. Values beyond mean ± 3 sd are grouped into one overflow bin per side.

In [ ]:
def vectors_of(r: dict) -> int:
    return r["num_vectors_vectorised"]


METRIC_LABEL = "vectors after vectorisation"
hist_path = SCAN_DIR / "histograms" / "num_vectors_vectorised.png"
hist_path.parent.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(7, 4))
h.plot_histogram_with_outliers([vectors_of(r) for r in page_records], ax, label=METRIC_LABEL)
ax.set_xlabel(METRIC_LABEL)
ax.set_ylabel("page count")
fig.tight_layout()
fig.savefig(hist_path, dpi=120, bbox_inches="tight")
plt.show()
plt.close(fig)
print("wrote", hist_path)

## 3. Sample `NUM_SAMPLES` distinct PDFs, evenly by rank

One page per PDF, every chosen page's vector count distinct, ascending so the cheap pages run first.

In [ ]:
samples = h.sample_distinct(h.cap_vectors(page_records, MAX_VECTORS), vectors_of, NUM_SAMPLES)
print(
    f"sampled {len(samples)} distinct PDF(s), one page each"
    + ("" if len(samples) == NUM_SAMPLES else f" (wanted {NUM_SAMPLES}: dataset ran out)")
)
for r in samples:
    print(f"  {Path(r['pdf_path']).name} page {r['page_index']}: {vectors_of(r)} vectors")
(RUN_DIR / "sample_pages.json").write_text(json.dumps(samples, indent=2), encoding="utf-8")

## 4. Run the classification-only report on the sample

`REPORT_CONFIG_PATH` is the template; the sample's PDFs, pages and an output folder are filled in. It runs in a subprocess; stopping it
partway (or a page running out of memory) still leaves finished pages on disk, so section 5 plots whatever completed.

In [ ]:
config_dict = json.loads(Path(REPORT_CONFIG_PATH).read_text(encoding="utf-8"))
config_dict.pop("input_dir", None)
config_dict["input_files"] = list(dict.fromkeys(r["pdf_path"] for r in samples))
config_dict["pages"] = h.pages_by_stem(samples)
config_dict["output_root"] = str(RUN_DIR / "pipeline_reports")

report_config = ReportConfig(**config_dict)
generated_config_path = RUN_DIR / "generated_report_config.json"
generated_config_path.write_text(report_config.model_dump_json(indent=2), encoding="utf-8")
print(
    f"running generate_pipeline_report: p2={report_config.p2} p3={report_config.p3}, "
    f"{len(config_dict['input_files'])} PDF(s), {len(samples)} page(s)"
)

# A subprocess, so a page that runs out of memory ends the run, not this kernel.
proc = subprocess.run(
    [
        sys.executable,
        "-m",
        "scripts.generate_pipeline_report",
        "--config",
        str(generated_config_path),
    ],
    cwd=PROJECT_ROOT,
)
if proc.returncode != 0:
    _LOG.warning(
        "generate_pipeline_report exited with code %d -- plotting what was written", proc.returncode
    )

## 5. Time taken vs vector count

Timing comes from each sample's `dump.json` (`step_durations` + sub-steps, flattened by
`timing.flatten_page_timing`); `total` is the sum of the phases that ran.

In [ ]:
report_run_dir = sorted(Path(config_dict["output_root"]).glob(f"*__{generated_config_path.stem}"))[
    -1
]
print("report run dir:", report_run_dir)

for r in samples:
    stem, r["timing"] = Path(r["pdf_path"]).stem, None
    dump_path = report_run_dir / stem / "dump.json"
    if not dump_path.is_file():
        _LOG.warning("no dump.json for %s (page %d)", stem, r["page_index"])
        continue
    page_dump = next(
        (p for p in dump_io.load_dump(dump_path).pages if p.result.page.index == r["page_index"]),
        None,
    )
    if page_dump is not None:
        r["timing"] = (
            flatten_page_timing(
                page_dump.result.step_durations, page_dump.debug_durations, page_dump.sub_durations
            )
            or None
        )

timed = [r for r in samples if r["timing"]]
print(f"{len(timed)}/{len(samples)} sample(s) have timing data")
plot_dir = RUN_DIR / "timing_plots"
plot_dir.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(
    [vectors_of(r) for r in timed],
    [r["timing"]["total"] for r in timed],
    color=h.BLUE,
    s=60,
    edgecolor="white",
)
ax.set_xlabel(METRIC_LABEL)
ax.set_ylabel("vector classification time (s)")
ax.set_title(f"Classification time vs {METRIC_LABEL}")
fig.tight_layout()
fig.savefig(plot_dir / "time_vs_num_vectors_vectorised.png", dpi=120, bbox_inches="tight")
plt.show()
plt.close(fig)
print("wrote", h.write_timing_results(samples, RUN_DIR / "timing_results.txt"))